# Entendimento do negócio

In [5]:
# Importações
import ee
import requests
import re
import os
import urllib3
import zipfile
import shutil
import json
import gc

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

from glob import glob
from io import BytesIO
from shapely.geometry import box

print("OK")

AttributeError: module 'pyarrow' has no attribute '__version__'

In [ ]:
# Roda o comando no terminal : earthengine authenticate --auth_mode=notebook
ee.Authenticate()
ee.Initialize(project="spatial-yew-490017-r3")
print(ee.String("Hello from the Earth Engine servers!").getInfo())

In [ ]:
# mostrar todas as colunas
pd.set_option('display.max_columns', None)

# não quebrar a largura da tabela
pd.set_option('display.expand_frame_repr', False)

# opcional: aumentar a largura máxima exibida
pd.set_option('display.width', 1000)

In [ ]:
os.environ['GDAL_DATA'] = os.path.join(os.environ['CONDA_PREFIX'], 'Library', 'share', 'gdal')
os.environ['PROJ_LIB'] = os.path.join(os.environ['CONDA_PREFIX'], 'Library', 'share', 'proj')

# Entendimento dos dados

In [ ]:
FILES_DIR = "files"
os.makedirs(FILES_DIR, exist_ok=True)

In [ ]:
# Download UCs boundaries via WFS (INDE/ICMBio)

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

ucs_dir = os.path.join(FILES_DIR, "limites_ucs")
os.makedirs(ucs_dir, exist_ok=True)

uc_typename = "ICMBio:limiteucsfederais_a"
uc_output_path = os.path.join(ucs_dir, "limites_ucs.geojson")

if os.path.exists(uc_output_path):
    print(f"Already downloaded: {uc_output_path}")
else:
    wfs_url = (
        "https://geoservicos.inde.gov.br/geoserver/ICMBio/ows"
        f"?service=WFS&version=2.0.0&request=GetFeature"
        f"&typeName={uc_typename}&outputFormat=application/json"
    )
    response = requests.get(wfs_url, timeout=180, verify=False)
    response.raise_for_status()
    with open(uc_output_path, 'wb') as f:
        f.write(response.content)
    print(f"Saved: {uc_output_path}")

gdf_ucs = gpd.read_file(uc_output_path)

print("Shape:", gdf_ucs.shape)
print("Columns:", gdf_ucs.columns.tolist())
print("CRS:", gdf_ucs.crs)

In [ ]:
# Reprojetar UCs para CRS métrico 

METRIC_CRS = "EPSG:5880"  # SIRGAS 2000 / Brazil Polyconic
CELL_SIZE_M = 5000        # 5km x 5km, ajustável

gdf_ucs_metric = gdf_ucs.to_crs(METRIC_CRS)
gdf_ucs_metric['geometry'] = gdf_ucs_metric.geometry.buffer(0)

print("Geometrias inválidas em gdf_ucs_metric:", (~gdf_ucs_metric.is_valid).sum())

In [ ]:
# INPE hotspots

INPE_YEARS = [2003, 2025]  

inpe_years_to_download = (
    INPE_YEARS if len(INPE_YEARS) == 1
    else list(range(min(INPE_YEARS), max(INPE_YEARS) + 1))
)
print("INPE years queued:", inpe_years_to_download)

In [ ]:
# INPE FOCO DE CALOR - downlaod de range de ano
inpe_dir = os.path.join(FILES_DIR, "inpe_focos")
os.makedirs(inpe_dir, exist_ok=True)

inpe_downloaded_files = {}

for year in inpe_years_to_download:

    csv_filename = f"focos_br_ref_{year}.csv"
    csv_path = os.path.join(inpe_dir, csv_filename)

    if os.path.exists(csv_path):
        inpe_downloaded_files[year] = csv_path
        continue

    zip_url = f"https://dataserver-coids.inpe.br/queimadas/queimadas/focos/csv/anual/Brasil_todos_sats/focos_br_todos-sats_{year}.zip"

    try:
        response = requests.get(zip_url, timeout=180)
        response.raise_for_status()

        tmp_dir = os.path.join(inpe_dir, f"tmp_{year}")
        os.makedirs(tmp_dir, exist_ok=True)

        with zipfile.ZipFile(BytesIO(response.content)) as z:
            z.extractall(tmp_dir)

        found_csvs = glob(os.path.join(tmp_dir, "**", "*.csv"), recursive=True)

        if found_csvs:
            shutil.move(found_csvs[0], csv_path)
            inpe_downloaded_files[year] = csv_path
        else:
            print(f"[{year}] No CSV found in zip")

        shutil.rmtree(tmp_dir, ignore_errors=True)

    except requests.exceptions.RequestException:
        print(f"[{year}] Download failed")

print("Done:", list(inpe_downloaded_files.keys()))

In [ ]:
# filtro contra a geometria das UCs
colunas_necessarias = [
    'latitude', 'longitude', 'data_pas', 'satelite', 'bioma',
    'estado', 'municipio', 'frp', 'risco_fogo', 'numero_dias_sem_chuva'
]

dtypes_reduzidos = {
    'satelite': 'category',
    'bioma': 'category',
    'estado': 'category',
    'municipio': 'category',
    'frp': 'float32',
    'risco_fogo': 'float32',
    'numero_dias_sem_chuva': 'float32',
    'latitude': 'float64',
    'longitude': 'float64',
}

inpe_df_list = []

for year, path in inpe_downloaded_files.items():

    df_year = pd.read_csv(path, usecols=colunas_necessarias, dtype=dtypes_reduzidos)

    # geometria temporária, só para o teste espacial deste ano — descartada logo em seguida
    points_year = gpd.GeoDataFrame(
        df_year[['latitude', 'longitude']],
        geometry=gpd.points_from_xy(df_year['longitude'], df_year['latitude']),
        crs="EPSG:4326"
    ).to_crs(METRIC_CRS)

    matched = gpd.sjoin(
        points_year,
        gdf_ucs_metric[['cnuc', 'geometry']],
        how='inner',
        predicate='within'
    )

    df_year_filtered = df_year.loc[matched.index].copy()
    df_year_filtered['file_year'] = year

    print(f"[{year}] Total: {len(df_year)} | Dentro de alguma UC: {len(df_year_filtered)}")

    inpe_df_list.append(df_year_filtered)

    del df_year, points_year, matched, df_year_filtered
    gc.collect()

df_inpe = pd.concat(inpe_df_list, ignore_index=True)
del inpe_df_list
gc.collect()

print("\nShape final (já filtrado por UC real):", df_inpe.shape)
print("Columns:", df_inpe.columns.tolist())
print("Years present:", sorted(df_inpe['file_year'].unique()))


In [ ]:
# AAF — download via ArcGIS FeatureServer (anos 2010-2026)

BASE_URL = "https://services3.arcgis.com/KYEMegXJrTiWSYWk/arcgis/rest/services"
PAGE_SIZE = 2000

aaf_urls_by_year = {}

# Padrão dinâmico — anos 2010 a 2022 seguem a mesma convenção de nome
for year in range(2010, 2023):
    aaf_urls_by_year[year] = f"{BASE_URL}/db_geo_compartilhado_dmif_fogo_aaf_{year}/FeatureServer/0"

# Exceções — nomes de serviço mudaram ano a ano a partir de 2023
aaf_urls_by_year[2023] = f"{BASE_URL}/AAF_2023_DGEO_ICMBIO_oficial/FeatureServer/0"
aaf_urls_by_year[2024] = f"{BASE_URL}/AAF_2024_DGEO_ICMBIO_oficial/FeatureServer/0"
aaf_urls_by_year[2025] = f"{BASE_URL}/AAF_2025_DGEO_oficial/FeatureServer/0"
aaf_urls_by_year[2026] = f"{BASE_URL}/AAF_2026/FeatureServer/0"

print("Total de anos mapeados:", len(aaf_urls_by_year))

In [ ]:
# Download com paginação, salvando um GeoJSON por ano

aaf_dir = os.path.join(FILES_DIR, "aaf")
os.makedirs(aaf_dir, exist_ok=True)

aaf_downloaded_files = {}

for year, base_url in aaf_urls_by_year.items():

    output_path = os.path.join(aaf_dir, f"aaf_{year}.geojson")

    if os.path.exists(output_path):
        aaf_downloaded_files[year] = output_path
        continue

    # Paginação — segue baixando enquanto o servidor retornar página cheia
    all_features = []
    offset = 0

    while True:
        query_url = (
            f"{base_url}/query?where=1%3D1&outFields=*&outSR=4326&f=geojson"
            f"&resultOffset={offset}&resultRecordCount={PAGE_SIZE}"
        )
        response = requests.get(query_url, timeout=120)
        response.raise_for_status()
        page_data = response.json()

        features = page_data.get('features', [])
        if not features:
            break

        all_features.extend(features)

        if len(features) < PAGE_SIZE:
            break

        offset += PAGE_SIZE

    # Salvar como GeoJSON válido
    final_geojson = {"type": "FeatureCollection", "features": all_features}

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(final_geojson, f)

    aaf_downloaded_files[year] = output_path
    print(f"[{year}] Salvo — {len(all_features)} registros")

print("\nAnos baixados:", list(aaf_downloaded_files.keys()))

In [ ]:
# Carregar todos os anos, checando consistência de colunas antes de concatenar

aaf_gdf_list = []
reference_columns = None

for year, path in aaf_downloaded_files.items():
    gdf_year = gpd.read_file(path)

    if reference_columns is None:
        reference_columns = set(gdf_year.columns)
    else:
        diff = set(gdf_year.columns).symmetric_difference(reference_columns)
        if diff:
            print(f"[{year}] Diferença de colunas em relação ao primeiro ano: {diff}")

    gdf_year['file_year'] = year
    aaf_gdf_list.append(gdf_year)

gdf_aaf = gpd.GeoDataFrame(pd.concat(aaf_gdf_list, ignore_index=True))


print("Shape:", gdf_aaf.shape)
print("Colunas:", gdf_aaf.columns.tolist())
print("Anos presentes:", sorted(gdf_aaf['file_year'].unique()))

In [ ]:
# Harmonizar colunas divergentes entre eras do schema

gdf_aaf["event_year"] = gdf_aaf["file_year"]

# converter cada fonte para datetime
date_from_old_schema = pd.to_datetime(gdf_aaf["data"], unit="ms", errors="coerce")
date_from_new_schema = pd.to_datetime(gdf_aaf["data_img"], errors="coerce")
gdf_aaf["event_date"] = date_from_old_schema.combine_first(date_from_new_schema)

#  recupera datas quando 'data' e 'data_img' estão ausentes
julian_fallback = pd.to_datetime(
    gdf_aaf["file_year"].astype(str), format="%Y", errors="coerce"
) + pd.to_timedelta(gdf_aaf["juliano"] - 1, unit="D")
gdf_aaf["event_date"] = gdf_aaf["event_date"].combine_first(julian_fallback)
gdf_aaf["event_month"] = gdf_aaf["event_date"].dt.month

# coalesce entre nomenclatura abreviada e completa
gdf_aaf["shape_area"] = gdf_aaf["Shape__Are"].combine_first(gdf_aaf["Shape__Area"])
gdf_aaf["shape_length"] = gdf_aaf["Shape__Len"].combine_first(gdf_aaf["Shape__Length"])

# Descartar eventos sem data válida
n_before = len(gdf_aaf)
gdf_aaf = gdf_aaf[gdf_aaf["event_date"].notna()].copy()

print(
    f"Eventos descartados por falta de data: {n_before - len(gdf_aaf)} ({(n_before - len(gdf_aaf)) / n_before * 100:.1f}%)"
)
print(f"Total de eventos final: {len(gdf_aaf)}")
print(f"Nulos em shape_area: {gdf_aaf['shape_area'].isna().sum()}")

# Preparação dos dados

In [2]:
# Reprojetar AAF 

gdf_aaf_metric = gdf_aaf.to_crs(METRIC_CRS)
print("CRS AAF após reprojeção:", gdf_aaf_metric.crs)


NameError: name 'gdf_aaf' is not defined

In [ ]:
gdf_aaf_metric['geometry'] = gdf_aaf_metric.geometry.buffer(0)
print("Geometrias inválidas em gdf_aaf_metric:", (~gdf_aaf_metric.is_valid).sum())

In [ ]:
# Descartar volume irrelevante
n_before = len(gdf_aaf_metric)
gdf_aaf_metric = gdf_aaf_metric[gdf_aaf_metric.is_valid].copy()

print(f"\nRegistros descartados por geometria inválida: {n_before - len(gdf_aaf_metric)}")
print(f"Total de eventos final: {len(gdf_aaf_metric)}")

In [ ]:
# Remover as poucas duplicatas geométricas reais 
n_before = len(gdf_aaf_metric)
gdf_aaf_metric = gdf_aaf_metric[
    ~gdf_aaf_metric.geometry.apply(lambda g: g.wkb).duplicated()
].copy()

print(f"Duplicatas geométricas removidas: {n_before - len(gdf_aaf_metric)}")
print(f"Total final: {len(gdf_aaf_metric)}")


In [ ]:
# Origem global fixa 

global_minx, global_miny, _, _ = gdf_ucs_metric.total_bounds

grid_records = []

for idx, row in gdf_ucs_metric.iterrows():
    uc_geom = row.geometry
    minx, miny, maxx, maxy = uc_geom.bounds

    i_start = int((minx - global_minx) // CELL_SIZE_M)
    i_end = int((maxx - global_minx) // CELL_SIZE_M) + 1
    j_start = int((miny - global_miny) // CELL_SIZE_M)
    j_end = int((maxy - global_miny) // CELL_SIZE_M) + 1

    for i in range(i_start, i_end):
        for j in range(j_start, j_end):
            cell_x = global_minx + i * CELL_SIZE_M
            cell_y = global_miny + j * CELL_SIZE_M
            cell = box(cell_x, cell_y, cell_x + CELL_SIZE_M, cell_y + CELL_SIZE_M)

            if uc_geom.intersects(cell):
                clipped = uc_geom.intersection(cell)
                if not clipped.is_empty:
                    grid_records.append({'cell_id': f"{i}_{j}", 'geometry': clipped})

gdf_grid_raw = gpd.GeoDataFrame(grid_records, crs=METRIC_CRS)
print("Peças de célula geradas:", len(gdf_grid_raw))

gdf_grid = gdf_grid_raw.dissolve(by='cell_id', as_index=False)
del gdf_grid_raw, grid_records
gc.collect()

print("Total de células globais únicas:", len(gdf_grid))

In [ ]:
# Tabela de associação de célula UC. 

cell_uc_membership = gpd.sjoin(
    gdf_grid,
    gdf_ucs_metric[['cnuc', 'geometry']],
    how='left',
    predicate='intersects'
)[['cell_id', 'cnuc']]

cells_per_uc_count = cell_uc_membership.groupby('cell_id').size()
print("Células pertencentes a mais de um UC:", (cells_per_uc_count > 1).sum())


In [ ]:
# Atribua eventos AAF a células da grade via sobreposição 

gdf_aaf_clipped = gpd.overlay(
    gdf_aaf_metric,
    gdf_grid[['cell_id', 'geometry']],
    how='intersection'
)

gdf_aaf_clipped['area_ha_cell'] = gdf_aaf_clipped.geometry.area / 10000

print("Eventos originais da AAF:", len(gdf_aaf_metric))
print("Linhas após a sobreposição:", len(gdf_aaf_clipped))



In [ ]:
n_before = len(df_inpe)
df_inpe = df_inpe.drop_duplicates(
    subset=['latitude', 'longitude', 'data_pas', 'satelite', 'frp']
).copy()
print(f"Duplicatas removidas de df_inpe: {n_before - len(df_inpe)}")
print(f"Total final: {len(df_inpe)}")

In [ ]:
# Filtro em cascata para o INPE 

gdf_inpe_points = gpd.GeoDataFrame(
    df_inpe,
    geometry=gpd.points_from_xy(df_inpe['longitude'], df_inpe['latitude']),
    crs="EPSG:4326"
).to_crs(METRIC_CRS)

gdf_inpe_with_cell = gpd.sjoin(
    gdf_inpe_points,
    gdf_grid[['cell_id', 'geometry']],
    how='inner',
    predicate='within'
)

print("Total de pontos:", len(gdf_inpe_points))
print("Total dentro de alguma célula:", len(gdf_inpe_with_cell))

del gdf_inpe_points
gc.collect()

In [ ]:
hotspots_uc = gdf_inpe_with_cell.merge(cell_uc_membership, on='cell_id', how='left')

print("Focos com UC associada:", hotspots_uc['cnuc'].notna().sum())
print("Focos sem UC associada:", hotspots_uc['cnuc'].isna().sum())

In [ ]:
# Persistir os datasets finais da Preparação 

OUTPUTS_DIR = os.path.join(FILES_DIR, "prepared")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Grade 
gdf_grid.to_parquet(os.path.join(OUTPUTS_DIR, "grid.parquet"))

# Associação célula <-> UC 
cell_uc_membership.to_parquet(os.path.join(OUTPUTS_DIR, "cell_uc_membership.parquet"))

# AAF já recortado por célula
gdf_aaf_clipped.to_parquet(os.path.join(OUTPUTS_DIR, "aaf_clipped.parquet"))

# INPE já associado à célula
gdf_inpe_with_cell.to_parquet(os.path.join(OUTPUTS_DIR, "inpe_with_cell.parquet"))

# UCs (necessário para nomes/atributos nas análises)
gdf_ucs_metric.to_parquet(os.path.join(OUTPUTS_DIR, "ucs_metric.parquet"))

print("Arquivos salvos em:", OUTPUTS_DIR)
for f in os.listdir(OUTPUTS_DIR):
    path = os.path.join(OUTPUTS_DIR, f)
    print(f"  {f}: {os.path.getsize(path) / 1e6:.1f} MB")

# Análise exploratória de dados (EDA)

# Pré-processamento

# Modelagem / treinamento

# Avaliação do modelo

# Ajuste de hiperparâmetros

# Validação final